In [ ]:
# Cell 00 (E046 frozen-input candidate): submit Version 5's settings (Humming dense, CUDA graphs <= 64, max-num-seqs
# 64, util 0.945; Kaggle KV 1.55M) + one rollout per task for tasks with several test inputs (ARC_MULTI_TEST) with a
# (1 + 0.5 (n - 1)) x output budget / turn cap (ARC_MULTI_BUDGET); 56 workers, KV gate 0.85.
# E128 on the 45 multi-pair eval tasks (E046 run 3): pass@2 83.1% vs 77.5% per pair at -29% tokens per task.
# ARC_CONTEXT_LIMIT = the server's max-model-len: turns never request past it (a 3-input task overflowed in run 3).
import json, os
for _k, _v in (("ARC_WORKERS", "56"),
               ("ARC_MULTI_TEST", "1"),
               ("ARC_MULTI_BUDGET", "0.5"),
               ("ARC_CONTEXT_LIMIT", "163840"),
               ("ARC_MAX_NUM_SEQS", "64"),
               ("ARC_GPU_UTIL", "0.945"),
               ("ARC_KV_ADMIT_MAX", "0.85"),
               ("ARC_KV_ADMIT_MIN_WORKERS", "16"),
               ("ARC_AUTO_REPAIR", "1"),
               ("ARC_ADMIT_MAX_WAITING", "1"),
               ("ARC_STALL_RESTART_S", "600"),
               ("ARC_CONN_HOLD", "1"),
               ("ARC_RESTART_RAMP", "16,8"),
               ("ARC_SERVER_EXTRA_ARGS", json.dumps(["--linear-backend", "humming", "--compilation-config",
                                                     json.dumps({"cudagraph_mode": "FULL",
                                                                 "max_cudagraph_capture_size": 64})]
                                                    + ["--prefix-cache-retention-interval", "512", "--kv-transfer-config", "{\"kv_connector\":\"OffloadingConnector\",\"kv_role\":\"kv_both\",\"kv_connector_extra_config\":{\"cpu_bytes_to_use\":34359738368,\"offload_prompt_only\":false,\"swa_store_policy\":\"checkpoint\",\"shared_region\":false,\"eviction_policy\":\"lru\"}}"]))):
    os.environ.setdefault(_k, _v)
print('PYTORCH_CUDA_ALLOC_CONF from the image:', repr(os.environ.get('PYTORCH_CUDA_ALLOC_CONF')), flush=True)
_ac = [x for x in os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '').split(',') if x and 'expandable_segments' not in x]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ','.join(_ac)
# The commit run ("Save & Run All") is a smoke test: server start + 3 tasks (2 multi-test). The competition rerun
# ignores ARC_COMMIT_MODE (cell 0 always runs "submit") and never sees the task list.
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    os.environ.setdefault("ARC_COMMIT_MODE", "smoke")
    os.environ.setdefault("ARC_TASK_IDS", "25ff71a9,27a28665,00576224")
    # smoke within 32 min of the first cell: model load, generation, force cut and final write all run
    os.environ.setdefault("ARC_STOP_NEW_S", "1620")
    os.environ.setdefault("ARC_FORCE_AT_S", "1740")
    os.environ.setdefault("ARC_HARD_STOP_S", "1800")
    os.environ.setdefault("ARC_KILL_AT_S", "1890")
    os.environ["ARC_TASK_IDS"] = "25ff71a9,27a28665,00576224,337b420f,25c199f5,12997ef3,2ccd9fef,0ca9ddb6,412b6263,351d6448,178fcbfb,137f0df0,2204b7a8,22eb0ac0,292dd178,05f2a901,2c737e39,230f2e48,20fb2937,3bdb4ada,2753e76c,20818e16,41e4d17e,13f06aa5"
    os.environ.setdefault("ARC_SMOKE_SANDBOXES", "56")
    _sx = json.loads(os.environ["ARC_SERVER_EXTRA_ARGS"])
    if "--num-gpu-blocks-override" not in _sx:
        os.environ["ARC_SERVER_EXTRA_ARGS"] = json.dumps(_sx + ["--num-gpu-blocks-override", "4500"])
    import shutil, subprocess, threading, time as _st
    os.environ.setdefault("ARC_SMOKE_ENV_LOG", "/kaggle/working/smoke_env.jsonl")

    def _smoke_env_sampler(path=os.environ["ARC_SMOKE_ENV_LOG"]):
        def _rd(p):
            try:
                return open(p).read().strip()
            except Exception:
                return None
        while True:
            try:
                rec = {"t": round(_st.time(), 1)}
                for k in ("memory.current", "memory.peak", "memory.max", "memory.swap.current"):
                    rec[k] = _rd(f"/sys/fs/cgroup/{k}")
                rec["stat"] = {l.split()[0]: int(l.split()[1]) for l in (_rd("/sys/fs/cgroup/memory.stat") or "").splitlines()
                               if l.split()[:1] and l.split()[0] in ("anon", "file", "shmem", "unevictable", "file_mapped")}
                rec["meminfo_kb"] = {l.split(":")[0]: int(l.split()[1]) for l in (_rd("/proc/meminfo") or "").splitlines()
                                     if l.split(":")[0] in ("MemTotal", "MemAvailable", "Shmem", "Mlocked", "Cached")}
                try:
                    rec["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=index,power.draw,power.limit,clocks.sm,"
                                                 "memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
                                                capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
                except Exception:
                    rec["gpu"] = []
                _sb_n, _sb_kb = 0, 0
                for _p in os.listdir("/proc"):
                    if _p.isdigit() and _rd(f"/proc/{_p}/oom_score_adj") == "1000":  # tool sandbox sessions
                        _sb_n += 1
                        _sb_kb += int(next((l.split()[1] for l in (_rd(f"/proc/{_p}/status") or "").splitlines()
                                            if l.startswith("VmRSS:")), 0))
                rec["sandbox"] = {"n": _sb_n, "rss_gib": round(_sb_kb / 2**20, 2)}
                rec["disk_free_gib"] = {d: round(shutil.disk_usage(d).free / 2**30, 1)
                                        for d in ("/kaggle/working", "/kaggle/temp", "/kaggle/tmp", "/tmp", "/dev/shm")
                                        if os.path.exists(d)}
                with open(path, "a") as f:
                    f.write(json.dumps(rec) + "\n")
            except Exception:
                pass
            _st.sleep(30)
    threading.Thread(target=_smoke_env_sampler, daemon=True).start()
print("E046 settings:", {k: os.environ.get(k) for k in ("ARC_WORKERS", "ARC_MULTI_TEST", "ARC_MULTI_BUDGET",
                                                     "ARC_CONTEXT_LIMIT", "ARC_KV_ADMIT_MAX", "ARC_MAX_NUM_SEQS", "ARC_GPU_UTIL",
                                                     "ARC_REASONING_EFFORT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT",
                                                     "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_ADMIT_MAX_WAITING",
                                                     "ARC_STOP_NEW_S", "ARC_FORCE_AT_S", "ARC_HARD_STOP_S", "ARC_KILL_AT_S",
                                                     "ARC_SERVER_EXTRA_ARGS", "PYTORCH_CUDA_ALLOC_CONF")}, flush=True)


In [ ]:
# Cell 0: clock, paths, fallback submission (before anything that can fail).
import glob, json, os, subprocess, sys, time
T0 = float(os.environ.get("ARC_T0") or time.time()); os.environ["ARC_T0"] = str(T0)
INPUT = os.environ.get("ARC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("ARC_OUT", "/kaggle/working"); os.environ["ARC_OUT"] = WORK
SCRATCH = os.environ.get("ARC_SCRATCH", "/tmp/dsv41"); os.makedirs(SCRATCH, exist_ok=True); os.makedirs(WORK, exist_ok=True)
RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
# Commit runs ("Save & Run All") only check the pipeline: "fallback" writes the fallback and stops; "smoke" runs
# ARC_TASK_LIMIT tasks with short deadlines. The competition rerun always runs "submit".
COMMIT_MODE = os.environ.get("ARC_COMMIT_MODE", "fallback")
MODE = "submit" if RERUN else os.environ.get("ARC_MODE", COMMIT_MODE)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# Every cache and HOME under SCRATCH, overriding the image's settings: with a model attached, /root/.cache is
# read-only on Kaggle (smoke v2 and bench v2: uv "Read-only file system" in /root/.cache/uv; setdefault kept the
# image's UV_CACHE_DIR). HOME is changed only for the subprocesses (install, solver, server), not this kernel.
_C = f"{SCRATCH}/cache"
SUB_HOME = f"{SCRATCH}/home"
os.makedirs(SUB_HOME, exist_ok=True)
for _k, _v in (("UV_CACHE_DIR", f"{_C}/uv"), ("XDG_CACHE_HOME", _C), ("VLLM_CACHE_ROOT", f"{_C}/vllm"),
               ("TRITON_CACHE_DIR", f"{_C}/triton"), ("TORCHINDUCTOR_CACHE_DIR", f"{_C}/inductor"),
               ("FLASHINFER_WORKSPACE_BASE", _C), ("TORCH_EXTENSIONS_DIR", f"{_C}/torch_ext"),
               ("HF_HOME", f"{_C}/hf"), ("CUDA_CACHE_PATH", f"{_C}/nv")):
    os.environ[_k] = _v
    os.makedirs(_v, exist_ok=True)
os.environ.pop("UV_NO_CACHE", None)
CHALLENGES = os.environ.get("ARC_CHALLENGES") or find_one(f"{INPUT}/**/arc-agi_test_challenges.json")
os.environ["ARC_CHALLENGES"] = CHALLENGES
_tasks = json.load(open(CHALLENGES))
_fb = {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in v["test"]] for t, v in _tasks.items()}
with open(f"{WORK}/submission.json.tmp", "w") as f:
    json.dump(_fb, f)
os.replace(f"{WORK}/submission.json.tmp", f"{WORK}/submission.json")
print(f"mode {MODE}; fallback submission for {len(_tasks)} tasks written at +{time.time() - T0:.0f}s", flush=True)
# The kit dataset is one tarball (Kaggle datasets do not keep nested folders reliably): unpack and check sha256.
KIT = os.environ.get("ARC_KIT")
if not KIT:
    try:
        import hashlib, tarfile
        _tgz = find_one(f"{INPUT}/**/dsv41_kit.tgz")
        KIT = f"{SCRATCH}/kit"
        with tarfile.open(_tgz) as tf:
            tf.extractall(KIT)
        _man = json.load(open(f"{KIT}/MANIFEST.json"))
        _bad = [f for f, h in _man.items() if hashlib.sha256(open(f"{KIT}/{f}", "rb").read()).hexdigest() != h]
        assert not _bad, f"kit files differ from MANIFEST: {_bad[:5]}"
        print(f"kit unpacked from {_tgz}: {len(_man)} files, sha256 ok", flush=True)
    except Exception as exc:  # without the kit nothing else can run: keep the fallback submission
        print("kit unavailable, fallback only:", type(exc).__name__, exc, flush=True)
        MODE = "fallback"
MODEL_SRC = os.environ.get("ARC_MODEL_SRC") or (os.path.dirname(find_one(f"{INPUT}/**/e043_manifest.json") or "") or None)
print("kit", KIT, "| model", MODEL_SRC, flush=True)

In [ ]:
# Cell 0b (frozen inputs from the attached pins dataset). The candidate's pinned solver, harness, helper modules and
# vLLM overlay come from a small private dataset (dsv41_pins.tgz + SHA256SUMS) built by
# `build_frozen_mt.py --pins-dataset`; the kit and model inputs stay Version 5's. After cell 0 has unpacked and checked
# the kit, this cell backs up the kit files the pins replace, installs the pins, and checks hashes, required features
# and settings.
# Commit run ("Save & Run All"): any failure RAISES, so the commit errors and the log shows "E046 CHECK FAILED".
# Competition rerun: the same failure restores the kit's v3 files and runs per pair at 36 workers (a scored submission
# beats an error); the rerun path never raises (S10). Uses cell 0's KIT, MODE, RERUN, INPUT and find_one.
import hashlib
import json
import os
import shutil
import tarfile

PINS = json.loads(r"""{"archive": "dsv41_pins.tgz", "archive_sha": "612e178ed98a3cc4e5b1f6989aaada4a8f59387fc827b607c58cb87d18f84355", "dataset": "dsv41-candb1-pins", "expect": {"ARC_ADMIT_MAX_WAITING": "1", "ARC_AUTO_REPAIR": "1", "ARC_CONN_HOLD": "1", "ARC_CONTEXT_LIMIT": "163840", "ARC_KV_ADMIT_MAX": "0.85", "ARC_MULTI_BUDGET": "0.5", "ARC_MULTI_TEST": "1", "ARC_RESTART_RAMP": "16,8", "ARC_STALL_RESTART_S": "600", "ARC_WORKERS": "56"}, "fallback_drop": ["ARC_MULTI_TEST", "ARC_MULTI_BUDGET", "ARC_KV_ADMIT_MAX", "ARC_CONTEXT_LIMIT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT", "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_REASONING_EFFORT", "ARC_ADMIT_MAX_WAITING"], "fallback_server_extra": "[\"--linear-backend\", \"humming\", \"--compilation-config\", \"{\\\"cudagraph_mode\\\": \\\"FULL\\\", \\\"max_cudagraph_capture_size\\\": 64}\"]", "harness_need": ["--multi-test", "--multi-budget", "--context-limit", "class ContextFull", "import builtins as _b", "def answer_items", "def context_budget"], "new": ["repo/evals/auto_repair.py", "repo/evals/auto_repair_names.json"], "opt_need": {"repo/evals/run_deepseek_tools.py": ["--auto-repair"], "solver_dsv41.py": ["ARC_ADMIT_MAX_WAITING", "ARC_STALL_RESTART_S", "ARC_CONN_HOLD", "ARC_RESTART_RAMP"]}, "sha": {"repo/evals/auto_repair.py": "2292369c4ad26fea4e2b3dc5e82dad31d1180192510484a4a1021a7e89b8a33d", "repo/evals/auto_repair_names.json": "571f47eb1bd077358d2153a943d0e6710c4ec18cf5828174d86c8be1fac03e92", "repo/evals/run_deepseek_tools.py": "882c817f2c93dd8e9ebf65c241a694549c5b7679cbacca5d16906a28dd62eecc", "repo/evals/train_check.py": "8fe361752010bba7599b24d83c4491ee462a2cffb1d9b1d018c67b90901fc982", "solver_dsv41.py": "dee879cea9993d3bd87112361c62ee17d9ae31143ec1fdaa8b4d3d36a8d03c5a", "vllm_overlay_rw.sha256": "624e7754fc322e76416425c9a3660306d137e9d1c43db444f766c3f4e117f04e", "vllm_overlay_rw.tgz": "1a3a667904d435964dd4ce1f9b5c205e2b148411ffa3b8f5e00d9527cda03306"}, "solver_need": ["MULTI_TEST", "MULTI_BUDGET", "CONTEXT_LIMIT", "KV_ADMIT_MAX", "split_multi", "restart_allowed", "shutdown_socket"]}""")  # filled in by build_frozen_mt.py: hashes, expected settings, feature checks


def pins_sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def pins_install(kit, archive, pins, state):
    """Back up each kit file the pins replace (files new to the kit have no v3 original), then extract the pins over
    the kit. `state` records the backups as they are made and whether extraction started, so a failure part-way
    leaves the fallback exactly what it needs (S10: a backup counts only once its copy exists)."""
    for member in pins["sha"]:
        if member not in pins["new"]:
            shutil.copy2(f"{kit}/{member}", f"{kit}/{member}.v3orig")
            state["backups"][member] = f"{kit}/{member}.v3orig"
    state["extracted"] = True
    with tarfile.open(archive) as tf:
        tf.extractall(kit)


def pins_check(kit, pins):
    """Raise AssertionError at the first mismatch: member hashes, the features every candidate's solver and harness
    need, the features its opt-in settings rely on, and the settings themselves."""
    for member, sha in pins["sha"].items():
        assert pins_sha256(f"{kit}/{member}") == sha, f"pinned file mismatch: {member}"
    with open(f"{kit}/solver_dsv41.py") as f:
        solver = f.read()
    with open(f"{kit}/repo/evals/run_deepseek_tools.py") as f:
        harness = f.read()
    for need in pins["solver_need"]:
        assert need in solver, f"solver lacks {need}"
    for need in pins["harness_need"]:
        assert need in harness, f"harness lacks {need}"
    for member, needs in pins["opt_need"].items():
        with open(f"{kit}/{member}") as f:
            text = f.read()
        for need in needs:
            assert need in text, f"{member} lacks {need}"
    for key, value in pins["expect"].items():
        assert os.environ.get(key) == value, f"{key}={os.environ.get(key)!r}, expected {value!r}"


def pins_fallback(kit, state, pins):
    """Competition rerun after a failed check: the kit's v3 files back (when the pins were extracted at all), Version
    5's settings (per pair, 36 workers, its server flags, every opt-in lever dropped). Never raises; returns what could
    not be restored."""
    kept = []
    try:
        if state["extracted"]:
            for member in pins["sha"]:
                if member in pins["new"]:
                    continue  # new to the kit: the v3 code never imports it
                try:
                    shutil.copy2(state["backups"][member], f"{kit}/{member}.restore")
                    os.replace(f"{kit}/{member}.restore", f"{kit}/{member}")
                except Exception as exc:
                    kept.append(f"{member} ({type(exc).__name__})")
        # (a pinned solver kept by a failed restore would pass the opt-in harness flags to the v3 harness, which exits)
        for key in pins["fallback_drop"]:
            os.environ.pop(key, None)
        os.environ["ARC_WORKERS"] = "36"
        os.environ["ARC_SERVER_EXTRA_ARGS"] = pins["fallback_server_extra"]  # no flag that needs the new overlay
    except Exception as exc:
        kept.append(f"{type(exc).__name__}: {exc}")
    return kept


if MODE != "fallback":
    _pins_state, _pins_why = {"backups": {}, "extracted": False}, ""
    try:
        assert KIT, "kit not unpacked"
        _pins_archive = find_one(f"{INPUT}/**/{PINS['archive']}")
        assert _pins_archive, f"{PINS['archive']} is not attached (dataset {PINS['dataset']})"
        assert pins_sha256(_pins_archive) == PINS["archive_sha"], f"{_pins_archive} is not the build's archive"
        pins_install(KIT, _pins_archive, PINS, _pins_state)
        pins_check(KIT, PINS)
    except Exception as exc:
        _pins_why = f"{type(exc).__name__}: {exc}"
    if not _pins_why:
        print("E046 CHECK OK: pinned files installed and checked:", {m: h[:8] for m, h in PINS["sha"].items()},
              PINS["expect"], flush=True)
    else:
        print("=" * 100, f"E046 CHECK FAILED: {_pins_why}", "=" * 100, sep="\n", flush=True)
        if not RERUN:
            raise RuntimeError(f"E046 CHECK FAILED (commit run): {_pins_why}")
        _pins_kept = pins_fallback(KIT, _pins_state, PINS)
        print("E046 FALLBACK: the kit's v3 code, per pair, 36 workers"
              + (f"; not restored, kept as is: {', '.join(_pins_kept)}" if _pins_kept else ""), flush=True)


In [ ]:
# Cell 1: stage the checkpoint to local disk in the background (stage_model.py), while vLLM installs.
# "warm" (read the mount once, no writes): smoke v2 (2026-10-24) with "copy" hit "Read-only file system" on the
# root overlay during the install while 131 GB were being written to /tmp.
STAGE_MODE = os.environ.get("ARC_STAGE_MODE", "warm")
STAGE_DONE = f"{SCRATCH}/stage_done"
if MODE != "fallback" and MODEL_SRC:
  try:
    if os.path.exists(STAGE_DONE):
        os.remove(STAGE_DONE)
    _stage = subprocess.Popen([sys.executable, "-I", f"{KIT}/stage_model.py", MODEL_SRC, f"{SCRATCH}/model", STAGE_DONE,
                               STAGE_MODE, os.environ.get("ARC_STAGE_THREADS", "16")],
                              stdout=open(f"{SCRATCH}/stage.log", "w"), stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"staging ({STAGE_MODE}) started, pid {_stage.pid}", flush=True)
  except Exception as exc:  # the solver then loads from MODEL_SRC (no staging marker)
    STAGE_DONE = ""
    print("staging failed to start:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 1b (smoke, commit runs only): input integrity, in a background thread so it costs no wall time. Checks every
# input the competition rerun reads: the competition JSON files parse; the kit (already sha256-checked by cell 0); the
# offline wheel set against its SHA256SUMS when it ships one; and every file of the model (136 files, 131 GB) by size
# and sha256 against the manifest made before upload. The model is hashed once the server is up, so it does not slow
# the load. Results go to /kaggle/working/smoke_integrity.json; the report cell prints them and fails the commit run on
# any mismatch. The competition rerun skips this cell. Uses cell 0's T0, INPUT, WORK, KIT, MODEL_SRC, MODE, RERUN and
# find_one.
import hashlib
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor

INTEGRITY = json.loads(r"""{"model": {"LICENSE": [1084, "f2c6c602815669d292889e5be8c802f2ed950653b77999b1584e8e6aed25d040"], "README.md": [4232, "80ea4ac33d69c27c3a3a51692ccbeb745ac5c71c8cfa1de6737492f9b2ef54e7"], "config.json": [5193, "f85a7c94bf42b4e25e16ebadb2de50480d41fed054e8b908e0d0fc284534bed4"], "e043_manifest.json": [40555, "0203d5074bad368e3e90195cb8132a7e70fb393f01e70a3075f97af13813e536"], "encoding/README.md": [12164, "a34a5ca98c65e5b5b2bedd693cc2e18e732fe50192723fb6937ae07219c23739"], "encoding/encoding.py": [37962, "13231c592e958fb39e2af6001dae69d90e7a36b265702b9b390236fb387599e2"], "encoding/test_encoding.py": [22449, "288b79c9ea7d018ae03b0dd3e54a8220cc3a2df1ff97f59907e75ed8112765d6"], "encoding/tests/test_input_1.json": [2761, "6a5a86694e47169c8f0c9243a7e24cc5db05447be0d3d3a9891034e913c518de"], "encoding/tests/test_input_2.json": [527, "fd985cda88e8640b8e2404d8e659f7006cad204eb41f73c9e6662b1f218c01aa"], "encoding/tests/test_input_3.json": [2554, "4207613f15c79b95ad92074bdb8b2b81c5ce9cfdf15b7a67f58ded6bb3ec4fa5"], "encoding/tests/test_input_4.json": [712, "4c3890a7bc042ec6396d98d024a39a73cdcffbe1d0253e852da0c35e65296f3e"], "encoding/tests/test_input_5.json": [1143, "88289b0aac8d650aa55c4ffeb392818fff3a0ac83c2293ddfd90546804d5e5c4"], "encoding/tests/test_output_1.txt": [2476, "91f79aa955717b414a82aabfb38bcd83ab5a998bd47db89d2a26338e8d3de550"], "encoding/tests/test_output_2.txt": [294, "9fa162587b73ab9b315be5c88d433f9e0e681b4b36926952f289099ed9bc1dab"], "encoding/tests/test_output_3.txt": [2481, "cf2b801d0738c88abb691bf4a096000c0e0737faab8a54a4ba5243c5513a3711"], "encoding/tests/test_output_4.txt": [574, "94433b68370dcc69fa61efc5011b5d0e28685e4716fee1ee77fbbfc0c9935b54"], "encoding/tests/test_output_5.txt": [408, "4d673091746553d1cb2e40a3fc01dc09a7c53c40627096a3e49c97a269a81f64"], "gptq/layer_00.json": [34363, "8070c088054bf4b1430486d8745e523c24ace9b2461a89fee4c6e84f08d2cf78"], "gptq/layer_00.safetensors": [1238726280, "0478e50eddbc304ee75c109647f2ac7847844689d2d267c4704759cfeba676d9"], "gptq/layer_01.json": [34308, "f924c25dc7fa0801bd0280f3730a9ea3075f30a0e2039c59b598be899d5c4da6"], "gptq/layer_01.safetensors": [1238726280, "3a306a2be5a069efe28c938d2b6e62dc8b2390d1d8f217c2bdb2ae60bd8f5f53"], "gptq/layer_02.json": [34337, "0e4bff8fc8f9b118c623a5eaaf4f1ae2dc7890a41bb45ce6f73632f2dcb8f1f9"], "gptq/layer_02.safetensors": [1238726280, "064c0760d7d803625ce1e7ce7b63de4e42ad7b289b42a8b64f968ff477e94f46"], "gptq/layer_03.json": [34308, "8e9fc5d8a0c95475b1965ebb03a8581d3ea73dc294508fb6a2fe3064028cd3df"], "gptq/layer_03.safetensors": [1238726280, "f1765866a9a98e09199363ddbf7e2741e8d20af2fa87e66f4b3e0a4ea0f6c9e6"], "gptq/layer_04.json": [34284, "7c02de8a310e4dfb6b846487096a256b5585925eecfd0e1f53959e28f92008c6"], "gptq/layer_04.safetensors": [1238726280, "1f83ee0c0ea45ee5fadb53bf2efbdac9cc1c85fe964b47ad59060e25b3cacf3a"], "gptq/layer_05.json": [34245, "4b210e637e46f7a606c8217162b3843e3c917e1880d904b357add4c06777bc17"], "gptq/layer_05.safetensors": [1238726280, "1f586d333951528d4bf1a0a65315578acf2cd6caea78ecf6b5566362bc639c86"], "gptq/layer_06.json": [34295, "84257b1f05474d6c06831e5f7a9f203af207734c4332a8400159d434d9c37bc7"], "gptq/layer_06.safetensors": [1238726280, "52bb959b015f1451c1fa604fc72e143754a4390982650489d3c16ce3e1e00ab3"], "gptq/layer_07.json": [34236, "e6c3ba1e042c12564082e67d25e693fa1dd5cd0d6518a4efae9811afec10ad8e"], "gptq/layer_07.safetensors": [1238726280, "6fe0738fd41e59d6b3ed45dd91711d29a16317fd553d4354c1041fa06e09776f"], "gptq/layer_08.json": [34259, "8ee6d15c002061526cd75bb6e06940a2281750b16d3f7dafb7dc64e09a7e369b"], "gptq/layer_08.safetensors": [1238726280, "fa2f89c9953d62cb50a210abc5ac9b99944b61396f65d1a917c3a765e52e7f1b"], "gptq/layer_09.json": [34274, "653bbd7b04ace39e1243880f0d0a106dbeb3c147b8cf82f121cadb8d8d8fda56"], "gptq/layer_09.safetensors": [1238726280, "1d58f087f9f210e4a9521f6f2dacba298f67e1e52e54f8751d8bac72b539b736"], "gptq/layer_10.json": [34251, "1f275848abcb0997eab0a157fca49218f67e07a51538965acc4d1c13dd4cbf5a"], "gptq/layer_10.safetensors": [1238726280, "2c1f30fb6ac72a5da1b728a720ec07b9ef405d05b210a6c2aee66aa1525fac61"], "gptq/layer_11.json": [34219, "67c3369fa913a2551eb285220cd58e80d6d967440b5d23d48eed3d6d23c29499"], "gptq/layer_11.safetensors": [1238726280, "494098074b791e1f98aae982211f510f27a83214ddb8dbab5a866adef50c6be6"], "gptq/layer_12.json": [34203, "f1e22cd9ddb9a2673298c5895f8b119429ef3228e4dc408ec122813b4bdd989e"], "gptq/layer_12.safetensors": [1238726280, "1475a262128907a0ea07537c73d3ed084b1c05241cbb0299c52b09d5ee1c5b1b"], "gptq/layer_13.json": [34227, "8b5fb5922b631dd98d3c163eed5fdb8f96214b026c56a2a927abd370d3b27eda"], "gptq/layer_13.safetensors": [1238726280, "0671ef29a3be767714bd44f674dd728a1d0856823ae9612e1a72c6c6fc1e2fce"], "gptq/layer_14.json": [34231, "90686ff97447f0b7ad3cb610e7893166d63366580577abfe65ae7923b3816999"], "gptq/layer_14.safetensors": [1238726280, "3f2ba1bd776811e8cfcf4eebcff2561fec7b9fe3c3ed9297e3f348f0a9af6041"], "gptq/layer_15.json": [34237, "f4574048fd6223c4f5b3f1350cc664e2e00d411541e013d64ea6ff862cdceb7e"], "gptq/layer_15.safetensors": [1238726280, "679b0c96c7d2ffee56bdc69c49eb27c85a2900141de1cf9ab2634c9932712acf"], "gptq/layer_16.json": [34199, "9cca7ea1727a2010def8d4ba070c9dc56b538c83c7c84dc7628fe7c2bbc8d771"], "gptq/layer_16.safetensors": [1238726280, "e292f2b6608fdc21af668abf3ce4504bd1d106ae727b658c75e0b2878c19fe4c"], "gptq/layer_17.json": [34190, "c09dd881c7b44791218b91ccddcf633fa36af6b91142c0b102acf5ee50d1ef6c"], "gptq/layer_17.safetensors": [1238726280, "1f6bd28a1de032aa1823865c50484066d2c27af7d757f8953e909f5e35ed761a"], "gptq/layer_18.json": [34215, "d5f5c809f786ef996a353debab33ba255e1c1543523704c6023978df9583a43e"], "gptq/layer_18.safetensors": [1238726280, "5222a1ed0d40563cc7ea6d8eb403d5e99a404f5c1476fda99b7469fff809f819"], "gptq/layer_19.json": [34227, "f36b03ed1be451a91113c5387f4443588ee2990d7b07ebe530b18fcbcd4e08e0"], "gptq/layer_19.safetensors": [1238726280, "144b7c9c51db9def462c05fdf469bd77a6bba6474ae65fce03993e4abaa18398"], "gptq/layer_20.json": [34216, "4f657799163609e1491cac8a424e2a330da61980f01d454904b8c5a41df6bd58"], "gptq/layer_20.safetensors": [1238726280, "174548e63b57648de373e8d78bec0c21558885d365bfbf38ed06a3b2318ff8fd"], "gptq/layer_21.json": [34203, "a98843417ed0d2065f08d406f279154da9981dfde9a80e8b02255c08427bf49f"], "gptq/layer_21.safetensors": [1238726280, "c8f057644083a2f01b0b121aaf7dcb6754907254b6dca32ac8b305a18ff46a10"], "gptq/layer_22.json": [34197, "d674cae5d82d88115561a36d50c9c6ce8ad3f2de609131ca038a2dd5f9892f34"], "gptq/layer_22.safetensors": [1238726280, "286ca8ee5a1f7cc92c90bf3e31f94d068eea09bb3da56f0cf5e08a642a9ee7dd"], "gptq/layer_23.json": [34234, "038b71cb3890fbaef35adf5314ecb5006681a2299342db94cb3b2258078edb28"], "gptq/layer_23.safetensors": [1238726280, "2c0d8f231a67e941d903e7f24d4fdef62747d5a53faff0daa02e6eb28c8cf659"], "gptq/layer_24.json": [34200, "30d38fbf20ea46de55d8da98358d9965ffe1ba2aa05df45e9a2d7eb24712dd7c"], "gptq/layer_24.safetensors": [1238726280, "d50027403a14ecc7ce46befed27b3d8d17c8f2aab20d078221e10a550a1d90cd"], "gptq/layer_25.json": [34177, "9fe1951e2ade0de1c881a893d079b816463ec99a64c9e40de04281f9b6d4b32a"], "gptq/layer_25.safetensors": [1238726280, "31d744cf34b945aeeb6159d612763a78694e031ffa7c88d28253ffa7d448f770"], "gptq/layer_26.json": [34167, "85b6007be2594c3633c86ac296d60357bf6d4546d29a861b0adc8df3b398702c"], "gptq/layer_26.safetensors": [1238726280, "950d25f1662eb5f94bbbb40aaa48efe64b614efce75552d3ef94956aff9e74bc"], "gptq/layer_27.json": [34196, "9af152d423070d986baeb82de18eb4081a4c63c06f78e60b7d3ee21193cefc27"], "gptq/layer_27.safetensors": [1238726280, "d197aa4c96ae2d8217d2e3876d6a7be01b07fd57c96458223848bfad17d3fc94"], "gptq/layer_28.json": [34177, "d31aea2be9ad421e0da3573caa8700bad75c29daa835fc5a030dbec1a836369a"], "gptq/layer_28.safetensors": [1238726280, "a05c5e62022ae4c5ed87c4d80c5b99f6b45e3c2cc57ade83346914b4ab58c590"], "gptq/layer_29.json": [34183, "7f6aae2fc4fff26b532eb4d304f21ff19d671f7b270a0aeb9bec5b898c8d1945"], "gptq/layer_29.safetensors": [1238726280, "8dfc815323471a19eea6be0fb4fa400c3609f2e63e151d41794682947536fb65"], "gptq/layer_30.json": [34165, "5f75bdeb910c779b8d0392b829662ad0b29e5e75268bcbf093ede29c8262139f"], "gptq/layer_30.safetensors": [1238726280, "84d0b70e0244235989679731649d65f803708c6e6447c09e0755d0dd1d33ecd1"], "gptq/layer_31.json": [34187, "a09043e40fe345082cfe8843cfe398274273f580f1dab2d25b01cc0328719d01"], "gptq/layer_31.safetensors": [1238726280, "05dd7b0e9e401c16b8e8f6432c3ea45cf01b9653aad8f6c682a210b0b6681dea"], "gptq/layer_32.json": [34186, "3f9c578268c9ad5682e4b10189a011192b762bdcab3c576d5e17322e5fe9074e"], "gptq/layer_32.safetensors": [1238726280, "f58e77c143b7bb549f449881c6e9f464859feaf7a4976d018b7426fcda1e063d"], "gptq/layer_33.json": [34145, "b4cbb0ebb96aa92cdd38f05aa08125294fa00bd6877a7b48b038d58979cf6b82"], "gptq/layer_33.safetensors": [1238726280, "4c2cf38a6d3b9fc61204112cea1db511ba4e4cd9869a8e00b9a97a72a1eec6ce"], "gptq/layer_34.json": [34159, "f55882c644eeb48dee59df10c99636f228097b8f31954ce5ce4f5f5a1e3d05ac"], "gptq/layer_34.safetensors": [1238726280, "af5691a717caf00971e4c29589bc9af992dde36f449be431484ae35a83102c8e"], "gptq/layer_35.json": [34229, "d9eba6ecb3c3338c2efdcc2df6bf4ebd7a4bad864ae4c294eec2e2a7ef7ddb4f"], "gptq/layer_35.safetensors": [1238726280, "1076ae8a2105848452b91e8b25d9490faf8be7ebe36ccef41d5c586191d20820"], "gptq/layer_36.json": [34229, "e9246f13555a3706a55cab4e181d3cf80c6c40050b3dc9b0873ec22504ce0825"], "gptq/layer_36.safetensors": [1238726280, "3f75926226d0ea0767c3f13984df112f2b3ba48c32c6d0bf87c3975ab7607521"], "gptq/layer_37.json": [34242, "bef7243980a9b24df76017d93f77b5c5c82ee17956883007045a8d0b3cff6097"], "gptq/layer_37.safetensors": [1238726280, "eec73875ba790e52cb889d35d38cfe2ab7d00e4758f04f39229ef62e4e6905be"], "gptq/layer_38.json": [34233, "bd2f44aedcc211f67dac637fabb891749166336eca6fba461fe0c3a6673eeab2"], "gptq/layer_38.safetensors": [1238726280, "f2999881df39f3e34e6b961c8dcd69602e6c2fa33e33867637207e6416ad7ae6"], "gptq/layer_39.json": [34321, "94a619f52806d0c9491959419007071278fcc7184806c75162863b09eef84a3d"], "gptq/layer_39.safetensors": [1238726280, "e0b9891b75d1238c669709d68424ac10cdd16b1e9f9fa89fc20cb847b976c80b"], "inference/README.md": [2029, "2834402823199ee24e9a42bdf36a0fc6daf94448f444cb062c042a057a798f1c"], "inference/config.json": [1982, "2e84f45cf1dac8c7fcbb200e96667d4b913275690668ed496f24c7747207a809"], "inference/convert.py": [9458, "035028340479145594a81d6084a8424e57363adf83c0d5983914783d95614d76"], "inference/engram.py": [8138, "11f35ecbead8150c35aa002b3d180ef290b05a25afe883a11884f94d476d3897"], "inference/examples/example.txt": [332, "8456a2f1d5a15b7aa1f697567e34f79af0522fb694df7ac789dae9853fff165d"], "inference/examples/example_harmony.json": [2156, "44ae84a3cbd360b1b8d303ed4006756afb77e5d014c4abb7c61ff9b9a094f6ea"], "inference/examples/images/carrots.jpeg": [212495, "5df896a4a07e127281c60fc957f8b3d73f4735b3258a0bf762b4383557f8fa9a"], "inference/examples/images/corn.jpeg": [56122, "251fe7ad58f44db139a6af5ed489e027eca93ae60878b83f3e709c7d6f0112a1"], "inference/generate.py": [8722, "8668d67f7d108e32b90d50cb0d8606889ceb2219bfe95741d84e22f70768e9f0"], "inference/image_processor.py": [7699, "482759e3bcc4e9bb5ee582b244cc563f5d0e163d8b48dda91ebb7106e62f9272"], "inference/kernel.py": [23790, "1236c3507019ed176f5dba5e04bcea58867cf654818c6cf138ed4845398c2455"], "inference/model.py": [61669, "e5decd76fe70fd56ef2f10124d245c6f64fb6a0003cc9068a1d3182a168b8d2d"], "inference/requirements.txt": [98, "be82bcd5775ccf8361ddd22cac53ef7b3b83434697fc3c61c75870450fb9437e"], "inference/run.sh": [1759, "91eddd7895f9fcd9bace53c21128c506e741e770f3c1a02464eff51f60401dfb"], "inference/vision.py": [4457, "5d49edc196a4ef22384abe76d35a40098cbe1e74b586c8f66a2edff4f076b26c"], "model-00001-of-00002.safetensors": [4990846656, "183eccf1215a92250c8102d1a757beef17a494a3e2c396729b7a84b53e857b9d"], "model-00002-of-00002.safetensors": [4751091632, "78cfb0103dd6918f9aca441c7876eede73faca3928b54f312c2d55efcb867ee1"], "model-engram-00001-of-00016.safetensors": [1088919216, "55c00ea48a06a67028a7c5eeb3897ae8e7f4119b0de0d1c2c8088b2674ded583"], "model-engram-00002-of-00016.safetensors": [4998951576, "fd43f6cc023ff267c37085ccecb074305d235fb3a23747d3bc99d22db6a81ef3"], "model-engram-00003-of-00016.safetensors": [4998951584, "3fa7839ef761a09e7937b78aa917c706570bd2309fb0a410104880a46cb3f05c"], "model-engram-00004-of-00016.safetensors": [4998951584, "1c897280d993b03794cd6d14712e2f995c5a683935e686d1f0d6c43920e1e13a"], "model-engram-00005-of-00016.safetensors": [4998951584, "d214812b29e538caaa0831ece23c636690699587763a9458ad44ce04bce513ef"], "model-engram-00006-of-00016.safetensors": [4998951584, "261a5fef67de77a7877395737f783b4adaadb7c9c0098019bd7cfda95afa4804"], "model-engram-00007-of-00016.safetensors": [4998951584, "f781ae4a959389211dbf92f4ad248c52c4de675d59d71f32a87aa59bb200809b"], "model-engram-00008-of-00016.safetensors": [4851701920, "c1d6d4993c9316a331464f9d07f97cbb45d6b68ff58dcb370e574a8a08bfecaa"], "model-engram-00009-of-00016.safetensors": [2529514248, "5de637f1aaecb973114866a6af06a67ffcafed5c7ab7f21a1746b93c86e5a66a"], "model-engram-00010-of-00016.safetensors": [4998951576, "5a5d3e532bf817a2753685f133ab1144b07c0771f0a5389d44033e62c656e13e"], "model-engram-00011-of-00016.safetensors": [4998951584, "3560c30b277c7cfcf826ba3d9ff38212fcdc7703f087a858f0ccdd914e3e427f"], "model-engram-00012-of-00016.safetensors": [4998951584, "041646d1e4c7ae4e4c8473504ed0777dda00278b3210532b449693964add675b"], "model-engram-00013-of-00016.safetensors": [4998951584, "5e9f091824203c027f2cdeba6d294de700dd2587b452a9cbfcdc97f81125f6f9"], "model-engram-00014-of-00016.safetensors": [4998951584, "32c53926732fa89c634147a5048c44221ed6851f7915f6db85045a1cebad3360"], "model-engram-00015-of-00016.safetensors": [4998951584, "e6c9a6bed470838bc472519603f53646e2b727c64b937a12b3ca2f2e67850851"], "model-engram-00016-of-00016.safetensors": [3334016448, "69b910b6c8e4c1c98d4a117b0cf844218f10e6b07b72346386e0e7752dd548b8"], "model.safetensors.index.json": [90861, "62c48a61621903219dfe74520feb887ec5bcef330b2027ae801d68d7a1ae7931"], "serving/apply_compact_engram.py": [14059, "fd3ef2f9d51eb2e44b572db783b0b384952b0e3319fed74c9e147cac5aa92f17"], "serving/apply_e043_humming.py": [10843, "991074c8f2a47834d98b369eee757b0e86a777802c811fc47e05f7bb522f787b"], "serving/e043_humming.py": [4780, "74dcbb141ff0ad29b56123f1ae24eef8e0249dea48a36dbe76d983a7fa2be8d8"], "tokenizer.json": [6367257, "c90dfa01249db1be4245780a052ede752e1361c612ac6d08e2bdada7d599476b"], "tokenizer_config.json": [801, "6ac8c8dc065ed118161d02dd532749ae3f52c243deac27872134fae2f50d8547"]}}""")  # filled in by build_frozen_mt.py: {"model": {path: [size, sha256]}}
INTEGRITY_OUT = f"{WORK}/smoke_integrity.json"


def integrity_save(state):
    with open(INTEGRITY_OUT + ".tmp", "w") as f:
        json.dump(state, f, indent=1)
    os.replace(INTEGRITY_OUT + ".tmp", INTEGRITY_OUT)


def integrity_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(16 << 20)
            if not block:
                return h.hexdigest()
            h.update(block)


def integrity_worker(state, model_dir, threads=8, wait_s=1800):
    try:
        state["competition"] = {}
        comp_files = sorted({p for p in (find_one(f"{INPUT}/**/arc-agi_test_challenges.json"),) if p})
        comp_dir = os.path.dirname(comp_files[0]) if comp_files else None
        for name in sorted(os.listdir(comp_dir)) if comp_dir else []:
            if name.endswith(".json"):
                with open(os.path.join(comp_dir, name)) as f:
                    state["competition"][name] = len(json.load(f))
        state["kit"] = "sha256-checked by cell 0 (MANIFEST.json)" if KIT else "MISSING"
        wheel = find_one(f"{INPUT}/**/vllm-0.28.1rc1*.whl")
        wdir = os.path.dirname(wheel) if wheel else None
        sums = os.path.join(wdir, "SHA256SUMS") if wdir else None
        if sums and os.path.exists(sums):
            bad = []
            lines = [l.split() for l in open(sums) if l.strip()]
            for digest, name in ((l[0], l[-1].lstrip("*")) for l in lines):
                p = os.path.join(wdir, name)
                if not os.path.exists(p) or integrity_sha256(p) != digest:
                    bad.append(name)
            state["wheels"] = {"checked": len(lines), "bad": bad}
        else:
            state["wheels"] = {"files": len(os.listdir(wdir)) if wdir else 0, "note": "no SHA256SUMS shipped; uv installs "
                               "from the pinned requirements.lock and the import check in cell 2 runs"}
        integrity_save(state)
        log = f"{WORK}/dsv41_logs/solver.log"
        while time.time() - T0 < wait_s:  # hash after the server is up, so the load does not compete for the mount
            if os.path.exists(log) and "server up" in open(log, errors="replace").read():
                break
            time.sleep(15)
        files = INTEGRITY["model"]
        res = {"files": len(files), "checked": 0, "missing": [], "size_mismatch": [], "sha_mismatch": [], "bytes": 0,
               "started_min": round((time.time() - T0) / 60, 1)}
        state["model"] = res
        t0 = time.time()

        def check(rel):
            size, digest = files[rel]
            p = os.path.join(model_dir, rel)
            if not os.path.exists(p):
                return rel, "missing", 0
            if os.path.getsize(p) != size:
                return rel, "size_mismatch", 0
            return rel, ("ok" if integrity_sha256(p) == digest else "sha_mismatch"), size

        with ThreadPoolExecutor(threads) as ex:
            for rel, verdict, n in ex.map(check, sorted(files, key=lambda r: -files[r][0])):
                res["checked"] += 1
                res["bytes"] += n
                if verdict != "ok":
                    res[verdict].append(rel)
                if res["checked"] % 10 == 0:
                    res["minutes"] = round((time.time() - t0) / 60, 1)
                    integrity_save(state)
        res["minutes"] = round((time.time() - t0) / 60, 1)
        state["done"] = True
    except Exception as exc:
        state["error"] = f"{type(exc).__name__}: {exc}"
    integrity_save(state)


if not RERUN and MODE != "fallback":
    _integrity_state = {"started_min": round((time.time() - T0) / 60, 1), "model_dir": MODEL_SRC}
    integrity_save(_integrity_state)
    threading.Thread(target=integrity_worker, args=(_integrity_state, MODEL_SRC or ""), daemon=True).start()
    print(f"input integrity check started in the background ({len(INTEGRITY['model'])} model files; hashed after the "
          "server is up)", flush=True)


In [ ]:
# Cell 2: offline install of the fork wheel set, the vLLM overlay, the CUDA 13 shim (E042's Kaggle recipe).
import hashlib, shutil, tarfile
V = os.environ.get("ARC_VENV", f"{SCRATCH}/venv"); PY = f"{V}/bin/python"; SP = f"{V}/lib/python3.12/site-packages"
BASE_ENV = dict(os.environ)
def sh(cmd, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors="replace", timeout=timeout, env=env)
    return r.returncode, (r.stdout or "") + (r.stderr or "")
if MODE != "fallback" and not os.environ.get("ARC_SKIP_INSTALL"):
  try:
    real = "/usr/local/nvidia/lib64"  # host driver libcuda first (the image also has an older compat libcuda)
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"
    WHEELS = os.path.dirname(find_one(f"{INPUT}/**/vllm-0.28.1rc1*.whl"))
    t = time.time()
    _uvc = os.environ["UV_CACHE_DIR"]
    _ienv = dict(os.environ, HOME=SUB_HOME, UV_CACHE_DIR=_uvc, TMPDIR=f"{SCRATCH}/tmp")
    os.makedirs(f"{SCRATCH}/tmp", exist_ok=True)
    if shutil.which("uv"):
        rc, out = sh(f"uv venv --cache-dir {_uvc} --python {sys.executable} {V} && uv pip install --cache-dir {_uvc} "
                     f"--python {PY} --no-index --no-deps --find-links {WHEELS} -r {WHEELS}/requirements.lock",
                     timeout=3600, env=_ienv)
    else:
        rc, out = sh(f"{sys.executable} -m venv {V} && {PY} -m pip install --no-cache-dir --no-index --no-deps "
                     f"--find-links {WHEELS} -r {WHEELS}/requirements.lock", timeout=3600, env=_ienv)
    print(f"offline install rc {rc} in {time.time() - t:.0f}s", out[-800:], flush=True)
    assert rc == 0, "vLLM install failed"
    with tarfile.open(f"{KIT}/vllm_overlay_rw.tgz") as tf:
        tf.extractall(SP)
    for line in open(f"{KIT}/vllm_overlay_rw.sha256"):
        h, rel = line.split()
        assert hashlib.sha256(open(f"{SP}/{rel}", "rb").read()).hexdigest() == h, f"overlay mismatch: {rel}"
    print("vLLM overlay installed and checked", flush=True)
    cu = f"{SP}/nvidia/cu13"; shim = f"{SCRATCH}/cuda13"
    os.makedirs(f"{shim}/lib64/stubs", exist_ok=True)
    for d in ("bin", "include", "nvvm", "lib", "share"):
        if os.path.exists(f"{cu}/{d}") and not os.path.lexists(f"{shim}/{d}"):
            os.symlink(f"{cu}/{d}", f"{shim}/{d}")
    for f in os.listdir(f"{cu}/lib"):
        if not os.path.lexists(f"{shim}/lib64/{f}"):
            os.symlink(f"{cu}/lib/{f}", f"{shim}/lib64/{f}")
    for name in ("cudart", "nvrtc"):
        r = sorted(glob.glob(f"{cu}/lib/lib{name}.so.*"))
        if r and not os.path.lexists(f"{shim}/lib64/lib{name}.so"):
            os.symlink(r[0], f"{shim}/lib64/lib{name}.so")
    drv = next(iter(sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so*")) + sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*"))), None)
    for dst in (f"{shim}/lib64/stubs/libcuda.so", f"{shim}/lib64/libcuda.so"):
        if drv and not os.path.lexists(dst):
            os.symlink(drv, dst)
    BASE_ENV = dict(os.environ)
    BASE_ENV.update(CUDA_HOME=shim, CUDA_PATH=shim, PATH=f"{V}/bin:{shim}/bin:" + os.environ["PATH"],
                    LD_LIBRARY_PATH=os.environ.get("LD_LIBRARY_PATH", "") + f":{shim}/lib64")
    rc, out = sh(f'nvcc --version | tail -1; {PY} -c "import torch, vllm, flashinfer, importlib.metadata as m; '
                 f'print(torch.__version__, torch.cuda.device_count(), vllm.__version__, flashinfer.__version__, '
                 f'm.version(\'humming-kernels\'))"', timeout=900, env=BASE_ENV)
    print("toolchain / imports:", rc, out[-600:], flush=True)
    rc, out = sh("free -g | head -2; df -h /tmp /kaggle/working | tail -2; nproc; cat /sys/fs/cgroup/memory.max "
                 "/sys/fs/cgroup/memory.oom.group 2>/dev/null; grep -m1 'model name' /proc/cpuinfo", timeout=60)
    print(out, flush=True)
  except Exception as exc:  # no working vLLM: keep the fallback submission
    print("install failed, fallback only:", type(exc).__name__, exc, flush=True)
    MODE = "fallback"
# The tool code runs on the image's Python (scipy etc.); if that cannot import numpy/scipy in the executor's
# environment (-I, PATH only), use the venv's Python instead (numpy only).
TOOL_PY = os.environ.get("ARC_TOOL_PYTHON", sys.executable)
if MODE != "fallback":
    try:
        r = subprocess.run([TOOL_PY, "-I", "-c", "import numpy, scipy.ndimage; print(numpy.__version__)"],
                           env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1"}, capture_output=True,
                           text=True, timeout=300)
        print("tool python", TOOL_PY, "->", r.returncode, (r.stdout + r.stderr).strip()[-300:], flush=True)
        if r.returncode != 0 and os.path.exists(PY):
            TOOL_PY = PY
    except Exception as exc:
        print("tool python check failed:", exc, flush=True)
BASE_ENV.pop("DSV41_SYNTH_INIT", None)  # the overlay's synthetic init must stay off for real weights
BASE_ENV["HOME"] = SUB_HOME  # /root/.cache is read-only on Kaggle with a model attached
# FlashInfer's sampling module cannot JIT on the Kaggle image (E042 K-runs); vLLM's PyTorch sampler instead.
BASE_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

In [ ]:
# Cell 2c (smoke, commit runs only): a deliberate engine hang, to exercise the failure handling on Kaggle. A background
# thread waits for "server up", then freezes this notebook's own VLLM::EngineCore process (SIGSTOP; found among this
# kernel's descendants, never by name alone). The API server keeps answering while nothing is generated, which is the
# hang vLLM itself does not detect. The solver's progress watchdog (S20) must mark the server dead and restart it; the
# re-launch hold (S21) and the post-restart ramp (S22) then run. The smoke shortens the watchdog to ARC_SMOKE_STALL_S and
# lets a restart happen late in the short run (RESTART_MIN_LEFT_S). If no restart happened after ARC_SMOKE_HANG_MAX_S,
# the engine is resumed (SIGCONT) so the smoke still ends, and the report cell fails the commit run. The competition
# rerun skips this cell. Uses cell 0's T0, WORK, MODE and RERUN and cell 2's BASE_ENV.
import json
import os
import signal
import threading
import time

HANG_OUT = f"{WORK}/smoke_hang.json"


def hang_save(state):
    with open(HANG_OUT + ".tmp", "w") as f:
        json.dump(state, f, indent=1)
    os.replace(HANG_OUT + ".tmp", HANG_OUT)


def hang_descendants(pid):
    kids = {}
    for p in os.listdir("/proc"):
        if p.isdigit():
            try:
                with open(f"/proc/{p}/stat") as f:
                    ppid = int(f.read().rsplit(")", 1)[1].split()[1])
                kids.setdefault(ppid, []).append(int(p))
            except (OSError, ValueError, IndexError):
                pass
    out, todo = [], [pid]
    while todo:
        for c in kids.get(todo.pop(), []):
            out.append(c)
            todo.append(c)
    return out


def hang_cmd(pid):
    try:
        with open(f"/proc/{pid}/cmdline", "rb") as f:
            return f.read().replace(b"\0", b" ").decode(errors="replace").strip()
    except OSError:
        return ""


def hang_log_has(path, text):
    try:
        with open(path, errors="replace") as f:
            return text in f.read()
    except OSError:
        return False


def hang_worker(state, after_s, max_s):
    log = f"{WORK}/dsv41_logs/solver.log"
    try:
        while time.time() - T0 < 3600:
            if hang_log_has(log, "server up"):
                break
            time.sleep(5)
        time.sleep(after_s)
        cores = [p for p in hang_descendants(os.getpid()) if hang_cmd(p).startswith("VLLM::EngineCore")]
        if not cores:
            state["error"] = "no VLLM::EngineCore among this kernel's descendants"
            return
        pid = cores[0]
        os.kill(pid, signal.SIGSTOP)
        state.update(engine_pid=pid, frozen_min=round((time.time() - T0) / 60, 2))
        hang_save(state)
        t0 = time.time()
        while time.time() - t0 < max_s:
            if hang_log_has(log, "server start (restart"):
                state["restart_seen_after_s"] = round(time.time() - t0)
                break
            time.sleep(5)
        else:
            os.kill(pid, signal.SIGCONT)  # the watchdog did not act: resume so the smoke can still end
            state["resumed_without_restart"] = True
    except Exception as exc:
        state["error"] = f"{type(exc).__name__}: {exc}"
    hang_save(state)


if not RERUN and MODE != "fallback":
    BASE_ENV["ARC_STALL_RESTART_S"] = os.environ.get("ARC_SMOKE_STALL_S", "60")
    BASE_ENV["ARC_RESTART_MIN_LEFT_S"] = os.environ.get("ARC_SMOKE_RESTART_MIN_LEFT_S", "300")
    _hang_state = {"after_s": int(os.environ.get("ARC_SMOKE_HANG_AFTER_S", "20"))}
    hang_save(_hang_state)
    threading.Thread(target=hang_worker, args=(_hang_state, _hang_state["after_s"],
                                               int(os.environ.get("ARC_SMOKE_HANG_MAX_S", "420"))), daemon=True).start()
    print(f"hang test armed: freeze VLLM::EngineCore {_hang_state['after_s']} s after the server is up; watchdog "
          f"{BASE_ENV['ARC_STALL_RESTART_S']} s", flush=True)


In [ ]:
# Cell 2b (smoke, commit runs only): the tool sandbox at full load, before the solver starts. Starts as many sandbox
# sessions as the solver runs workers (each its own process with a 4 GiB address-space cap, 64 MB file cap, nice 10 and
# oom_score_adj 1000, like solver.session_factory), runs a typical ARC cell in every one, then checks that an oversized
# allocation fails inside its session, that a runaway cell is interrupted and the session restarts, and that closing
# every session returns host memory. Commit run: a failure RAISES ("SANDBOX CHECK FAILED"); the competition rerun skips
# this cell. Uses cell 0's KIT, MODE and RERUN, and cell 2's TOOL_PY.
import os
import sys
import threading
import time


def sandbox_meminfo_gib():
    out = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, value = line.split(":", 1)
            if key in ("MemAvailable", "AnonPages", "Shmem"):
                out[key] = int(value.split()[0]) / 2**20
    return out


def sandbox_pss_gib(pids):
    total = 0
    for pid in pids:
        try:
            with open(f"/proc/{pid}/smaps_rollup") as f:
                total += sum(int(l.split()[1]) for l in f if l.startswith("Pss:"))
        except OSError:
            pass
    return total / 2**20


def sandbox_check(kit, python, sessions, mem_mb=4096, timeout=30, alloc_gib=6):
    """Returns (report lines, failures)."""
    sys.path.insert(0, f"{kit}/repo/evals")
    from arc_harness.executor import LocalExecutor
    train = [{"input": [[0, 1, 0], [1, 1, 1]], "output": [[1, 0, 1], [0, 0, 0]]}]
    test = [[1, 0, 0], [0, 1, 1]]
    cell = ("import numpy as np\nfrom scipy import ndimage\ng = np.array(test_input)\n"
            "lab, n = ndimage.label(g)\nprint(n, (1 - g).tolist())")
    lines, failures, results, lock = [], [], [], threading.Lock()
    before = sandbox_meminfo_gib()
    exs = [LocalExecutor(train, test, timeout, 16000, memory_mb=mem_mb, python=python, style="ipython",
                         cpu_seconds=7200) for _ in range(sessions)]

    def run(ex):
        t0 = time.time()
        try:
            r = ex.execute(cell)
            ok = r.get("process_status") in (None, "ok", "completed") and "[[0, 1, 1]" in (r.get("stdout") or "")
            with lock:
                results.append((ok, time.time() - t0, (r.get("stderr") or "")[-200:]))
        except Exception as exc:
            with lock:
                results.append((False, time.time() - t0, f"{type(exc).__name__}: {exc}"))

    threads = [threading.Thread(target=run, args=(ex,)) for ex in exs]
    t0 = time.time()
    for th in threads:
        th.start()
    for th in threads:
        th.join(300)
    pids = [ex.proc.pid for ex in exs if getattr(ex, "proc", None)]
    loaded = sandbox_meminfo_gib()
    good = sum(1 for ok, _, _ in results if ok)
    slow = max((dt for _, dt, _ in results), default=0)
    lines.append(f"{good}/{sessions} sessions ran a numpy/scipy cell in {time.time() - t0:.0f} s (slowest {slow:.0f} s); "
                 f"sandbox PSS {sandbox_pss_gib(pids):.2f} GiB; host anon {before['AnonPages']:.1f} -> "
                 f"{loaded['AnonPages']:.1f} GiB, MemAvailable {before['MemAvailable']:.1f} -> {loaded['MemAvailable']:.1f} GiB")
    if good < sessions:
        failures.append(f"{sessions - good} sessions failed: {[e for ok, _, e in results if not ok][:2]}")

    # an allocation over the session's cap must fail inside the session, which stays usable
    ex = exs[0]
    r = ex.execute(f"x = bytearray({alloc_gib} * 2**30)\nprint('allocated')")
    text = (r.get("stdout") or "") + (r.get("stderr") or "")
    contained = "allocated" not in text and ("MemoryError" in text or r.get("state_reset"))
    after = ex.execute("print(6 * 7)")
    lines.append(f"{alloc_gib} GiB allocation in one session: {'blocked' if contained else 'NOT blocked'} "
                 f"({text.strip().splitlines()[-1][:90] if text.strip() else r.get('process_status')}); session after: "
                 f"{(after.get('stdout') or '').strip() or after.get('process_status')}")
    if not contained or "42" not in (after.get("stdout") or ""):
        failures.append("memory cap did not contain the allocation or the session did not recover")

    # a runaway cell is interrupted at the timeout and the session keeps working
    ex = exs[1]
    t1 = time.time()
    r = ex.execute("while True:\n    pass")
    after = ex.execute("print(6 * 7)")
    lines.append(f"runaway cell: {r.get('process_status')} after {time.time() - t1:.0f} s; session after: "
                 f"{(after.get('stdout') or '').strip() or after.get('process_status')}")
    if "42" not in (after.get("stdout") or ""):
        failures.append("session did not recover after a timeout")

    for ex in exs:
        ex.close()
    time.sleep(3)
    alive = [p for p in pids if os.path.exists(f"/proc/{p}")]
    end = sandbox_meminfo_gib()
    lines.append(f"closed: {len(alive)} sandbox processes left; host anon {end['AnonPages']:.1f} GiB, "
                 f"MemAvailable {end['MemAvailable']:.1f} GiB")
    if alive:
        failures.append(f"{len(alive)} sandbox processes outlived close()")
    if end["AnonPages"] > before["AnonPages"] + 1.0:
        failures.append(f"host anon memory not returned ({before['AnonPages']:.1f} -> {end['AnonPages']:.1f} GiB)")
    return lines, failures


if not RERUN and MODE != "fallback":
    try:
        _sb_lines, _sb_fail = sandbox_check(KIT, TOOL_PY, int(os.environ.get("ARC_SMOKE_SANDBOXES", "56")),
                                            int(os.environ.get("ARC_EXEC_MEM_MB", "4096")),
                                            int(os.environ.get("ARC_SMOKE_SANDBOX_TIMEOUT", "30")))
    except Exception as exc:
        _sb_lines, _sb_fail = [], [f"{type(exc).__name__}: {exc}"]
    for _l in _sb_lines:
        print("sandbox check:", _l, flush=True)
    if _sb_fail:
        print("=" * 100, f"SANDBOX CHECK FAILED: {_sb_fail}", "=" * 100, sep="\n", flush=True)
        raise RuntimeError(f"SANDBOX CHECK FAILED: {_sb_fail}")
    print("SANDBOX CHECK OK", flush=True)


In [ ]:
# Cell 3: the solver (vLLM server, tool harness, deadlines, watchdog, incremental submission).
if MODE != "fallback" and MODEL_SRC:
  try:
    import signal, threading
    env = dict(BASE_ENV)
    env.update(ARC_KIT=KIT, ARC_MODEL_DIR=MODEL_SRC, ARC_STAGE_DONE=STAGE_DONE, ARC_VLLM=f"{V}/bin/vllm",
               ARC_TOOL_PYTHON=TOOL_PY, ARC_SCRATCH=SCRATCH)
    if MODE == "smoke":
        env.setdefault("ARC_TASK_LIMIT", "8")
        for k, v in (("ARC_STOP_NEW_S", "4800"), ("ARC_FORCE_AT_S", "5400"), ("ARC_HARD_STOP_S", "5700")):
            env.setdefault(k, v)
    p = subprocess.Popen([PY, "-u", f"{KIT}/solver_dsv41.py"], env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace", start_new_session=True)
    # Independent kill switch: whatever hangs inside the solver, its whole session dies at T0 + ARC_KILL_AT_S
    # (default 11h45; the solver's own hard stop is 11h30), so cell 4 and the notebook end before Kaggle's 12 h.
    def _kill():
        try:
            os.killpg(p.pid, signal.SIGKILL)
            print("kill switch: solver session killed", flush=True)
        except Exception:
            pass
    _timer = threading.Timer(max(0.0, T0 + float(os.environ.get("ARC_KILL_AT_S", 42300)) - time.time()), _kill)
    _timer.daemon = True
    _timer.start()
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    _timer.cancel()
    print("solver exit code", p.returncode, flush=True)
  except Exception as exc:
    print("solver cell failed:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 4: final check of submission.json; log archive.
import tarfile
def _ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
try:
    sub = json.load(open(f"{WORK}/submission.json"))
    assert set(sub) == set(_tasks), f"task ids differ ({len(sub)} vs {len(_tasks)})"
    for t, v in _tasks.items():
        assert len(sub[t]) == len(v["test"]), f"{t}: {len(sub[t])} rows for {len(v['test'])} test inputs"
        assert all(_ok(r["attempt_1"]) and _ok(r["attempt_2"]) for r in sub[t]), f"{t}: invalid grid"
    print("submission.json valid", flush=True)
except Exception as exc:
    print("submission.json invalid, writing the fallback:", exc, flush=True)
    json.dump(_fb, open(f"{WORK}/submission.json", "w"))
try:
    logs = f"{WORK}/dsv41_logs"
    if os.path.isdir(f"{logs}/traces"):
        with tarfile.open(f"{WORK}/dsv41_traces.tar.gz", "w:gz") as tf:
            tf.add(f"{logs}/traces", arcname="traces")
        shutil.rmtree(f"{logs}/traces", ignore_errors=True)
    for f in ("stage.log",):
        if os.path.exists(f"{SCRATCH}/{f}"):
            shutil.copy(f"{SCRATCH}/{f}", f"{logs}/{f}" if os.path.isdir(logs) else f"{WORK}/{f}")
except Exception as exc:
    print("log archive failed:", exc, flush=True)
print(f"done at +{(time.time() - T0) / 3600:.2f}h", flush=True)

In [ ]:
# Cell 5 (smoke report, commit runs only): peaks of host memory (cgroup, the 173 GiB limit), GPU power / clocks /
# memory / temperature and the lowest free disk, from the sampler started in the settings cell (every 30 s)
import json, os
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN") and os.environ.get("ARC_SMOKE_ENV_LOG"):
    try:
        _recs = [json.loads(l) for l in open(os.environ["ARC_SMOKE_ENV_LOG"]) if l.strip()]
        _gib = lambda v: round(int(v) / 2**30, 1) if v not in (None, "", "max") and str(v).isdigit() else v
        _mx = lambda xs: max([x for x in xs if isinstance(x, (int, float))], default=None)
        print(f"smoke report: {len(_recs)} samples over {(_recs[-1]['t'] - _recs[0]['t']) / 60:.1f} min", flush=True)
        print("  cgroup memory.max", _gib(_recs[-1].get("memory.max")), "GiB; peak memory.current",
              _mx([_gib(r.get("memory.current")) for r in _recs]), "GiB; memory.peak", _gib(_recs[-1].get("memory.peak")),
              "GiB; swap", _gib(_recs[-1].get("memory.swap.current")), flush=True)
        for k in ("anon", "shmem", "file", "unevictable"):
            print(f"  peak cgroup {k}", _mx([_gib(r.get("stat", {}).get(k)) for r in _recs]), "GiB", flush=True)
        _mi = [r.get("meminfo_kb", {}) for r in _recs]
        print("  host MemTotal", round(_mi[-1].get("MemTotal", 0) / 2**20, 1), "GiB; min MemAvailable",
              round(min(m.get("MemAvailable", 0) for m in _mi) / 2**20, 1), "GiB; peak Shmem",
              round(max(m.get("Shmem", 0) for m in _mi) / 2**20, 1), "GiB; peak Mlocked",
              round(max(m.get("Mlocked", 0) for m in _mi) / 2**20, 1), "GiB", flush=True)
        _g = {}
        for r in _recs:
            for line in r.get("gpu", []):
                f = [x.strip() for x in line.split(",")]
                if len(f) == 7:
                    d = _g.setdefault(f[0], {"draw": [], "limit": f[2], "sm": [], "mem": [], "total": f[5], "temp": []})
                    for k, i in (("draw", 1), ("sm", 3), ("mem", 4), ("temp", 6)):
                        try:
                            d[k].append(float(f[i]))
                        except ValueError:
                            pass
        for i, d in sorted(_g.items()):
            print(f"  GPU {i}: power limit {d['limit']} W, peak draw {_mx(d['draw'])} W, peak SM {_mx(d['sm'])} MHz, "
                  f"peak memory {_mx(d['mem'])} / {d['total']} MiB, peak {_mx(d['temp'])} C", flush=True)
        _df = {}
        for r in _recs:
            for k, v in r.get("disk_free_gib", {}).items():
                _df[k] = min(_df.get(k, v), v)
        print("  lowest free disk (GiB):", _df, flush=True)
        _sb = [r.get("sandbox", {}) for r in _recs]
        print("  tool sandboxes: peak", max((s.get("n", 0) for s in _sb), default=0), "processes,",
              max((s.get("rss_gib", 0) for s in _sb), default=0), "GiB RSS", flush=True)
    except Exception as exc:
        print("smoke report failed:", type(exc).__name__, exc, flush=True)
    try:  # which code paths the smoke exercised (solver log + server metrics)
        _lg = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/dsv41_logs"
        _m = [json.loads(l) for l in open(f"{_lg}/metrics.jsonl") if l.strip()]
        _s = open(f"{_lg}/solver.log").read()
        _host = sum((x.get("host_tok_s") or 0) * (x.get("window_s") or 0) for x in _m)
        print("smoke coverage:", {"KV max": max((x.get("kv") or 0) for x in _m), "server running max":
              max((x.get("srv_running") or 0) for x in _m), "preemptions": max((x.get("preemptions") or 0) for x in _m),
              "tokens restored from host (KV offload loads)": int(_host), "KV gate holds logged": _s.count("KV gate:"),
              "server restarts": _s.count("server start (restart"), "attempts done": _s.count(" stop="),
              "force cut": "force deadline:" in _s, "final write": "submission written (final)" in _s}, flush=True)
    except Exception as exc:
        print("smoke coverage failed:", type(exc).__name__, exc, flush=True)
    _integrity_bad = None
    _hp = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/smoke_hang.json"
    if os.path.exists(_hp):  # cell 2c: a deliberate engine hang; the solver's watchdog must restart the server
        _hg = json.load(open(_hp))
        try:
            _hs = open(f"{os.environ.get('ARC_OUT', '/kaggle/working')}/dsv41_logs/solver.log").read()
        except OSError:
            _hs = ""
        _after = _hs.split("server stalled", 1)[1] if "server stalled" in _hs else ""
        print("hang test:", _hg, "| watchdog fired:", "server stalled" in _hs, "| restarts:", _hs.count("server start (restart"),
              "| re-queued after the stall:", _after.count("re-queue "), "| attempts finished after the stall:",
              _after.count(" stop="), "| conn retries spent on a dead server:", _after.count("(2):") + _after.count("(3):"),
              flush=True)
        if _hg.get("error") or _hg.get("resumed_without_restart") or "server stalled" not in _hs:
            _integrity_bad = "HANG TEST FAILED: the watchdog did not restart the frozen server (see the line above)"
        else:
            print("HANG TEST OK: frozen engine detected and the server restarted", flush=True)
    _ip = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/smoke_integrity.json"
    if os.path.exists(_ip):  # cell 1b's background check of every input (model hashed after the server came up)
        import time as _it
        for _ in range(48):  # wait up to 4 min for the model hash to finish
            _ig = json.load(open(_ip))
            if _ig.get("done") or _ig.get("error"):
                break
            _it.sleep(5)
        _md = _ig.get("model", {})
        print("input integrity: competition files", _ig.get("competition"), "| kit:", _ig.get("kit"), "| wheels:",
              _ig.get("wheels"), "| model:", {k: _md.get(k) for k in ("files", "checked", "missing", "size_mismatch",
              "sha_mismatch", "started_min", "minutes")}, "| error:", _ig.get("error"), flush=True)
        _mbad = any(_md.get(k) for k in ("missing", "size_mismatch", "sha_mismatch"))
        if _ig.get("error") or not _ig.get("done") or _mbad or (_ig.get("wheels") or {}).get("bad"):
            _integrity_bad = "INPUT INTEGRITY FAILED or incomplete: see the line above"
        else:
            print("INPUT INTEGRITY OK: every model file matches its sha256", flush=True)
    if _integrity_bad:
        print("=" * 100, _integrity_bad, "=" * 100, sep=chr(10), flush=True)
        raise RuntimeError(_integrity_bad)
